In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 29. Unsupervised Learning: t-SNE (t-Distributed Stochastic Neighbor Embedding)

## Algorithm Category
**Type**: Unsupervised Learning - Dimensionality Reduction & Visualization  
**Complexity**: Medium-High  
**Use Case**: Non-linear dimensionality reduction for visualization of high-dimensional data

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand t-SNE and its mathematical foundation
- Implement t-SNE for data visualization
- Understand perplexity and its role in t-SNE
- Compare t-SNE with PCA
- Visualize high-dimensional data in 2D/3D
- Apply t-SNE to real-world problems

## Historical Context

t-SNE was developed by Laurens van der Maaten and Geoffrey Hinton in 2008:
- van der Maaten, L. & Hinton, G. (2008): "Visualizing Data using t-SNE"
- Extension of SNE (Stochastic Neighbor Embedding)
- Widely used for visualizing high-dimensional data

**Key Papers/References:**
- van der Maaten, L. & Hinton, G. (2008). "Visualizing Data using t-SNE"
- van der Maaten, L. (2014). "Accelerating t-SNE using Tree-Based Algorithms"

## When to Use t-SNE

t-SNE is appropriate when:
- You need to visualize high-dimensional data
- Data has non-linear structure
- You want to explore cluster structure
- Working with embeddings or feature vectors
- Need to understand data relationships
- Data has local structure to preserve

## Theory & Mechanics

### Mathematical Foundation

t-SNE preserves local neighborhood structure by modeling pairwise similarities.

**High-Dimensional Space (Gaussian):**
$$p_{j|i} = \frac{\exp(-||x_i - x_j||^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-||x_i - x_k||^2 / 2\sigma_i^2)}$$

$$p_{ij} = \frac{p_{j|i} + p_{i|j}}{2N}$$

**Low-Dimensional Space (t-Distribution):**
$$q_{ij} = \frac{(1 + ||y_i - y_j||^2)^{-1}}{\sum_{k \neq l} (1 + ||y_k - y_l||^2)^{-1}}$$

**Cost Function (KL Divergence):**
$$C = \sum_{i} \sum_{j} p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

**Key Differences from SNE:**
- Uses symmetric joint probabilities
- Uses t-distribution (heavy-tailed) in low-dimensional space
- Better at preserving global structure

### How It Works

1. **Compute similarities**: Calculate pairwise similarities in high-dimensional space
2. **Set perplexity**: Choose perplexity (related to number of neighbors)
3. **Initialize**: Random initialization in low-dimensional space
4. **Optimize**: Minimize KL divergence using gradient descent
5. **Iterate**: Update positions until convergence

### Key Hyperparameters

- **n_components**: Number of dimensions for embedding (usually 2 or 3)
- **perplexity**: Effective number of neighbors (typically 5-50)
  - Lower: Focus on local structure
  - Higher: Focus on global structure
- **learning_rate**: Step size for optimization (default: 200)
- **n_iter**: Maximum iterations (default: 1000)
- **random_state**: Seed for reproducibility

### Advantages

- Captures non-linear structure
- Excellent for visualization
- Preserves local neighborhoods
- Can reveal cluster structure
- Works well with high-dimensional data

### Limitations

- Computationally expensive (O(n²))
- Non-deterministic (results vary)
- Cannot be applied to new data (no transform)
- Sensitive to perplexity
- May not preserve global structure
- Slow for large datasets


## Implementation

Let's implement t-SNE for data visualization.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_iris,  # Iris flower dataset (4 features, good for t-SNE demo)
    load_digits,  # Digits dataset (64 features, high-dimensional)
    load_wine  # Wine dataset (13 features, medium-dimensional)
)
from sklearn.manifold import TSNE  # t-Distributed Stochastic Neighbor Embedding (non-linear dimensionality reduction)
from sklearn.preprocessing import StandardScaler  # Feature scaling (important for t-SNE)
from sklearn.decomposition import PCA  # Principal Component Analysis (use before t-SNE for high-D data)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Iris Classification
# ============================================

# load_iris() loads the Iris flower dataset from scikit-learn
# This dataset has 4 features, which we'll reduce to 2 using t-SNE
iris = load_iris()  # Returns a Bunch object with data, target, feature_names
X = iris.data  # Features: flower measurements (150 samples × 4 features)
y = iris.target  # True labels: flower species (for visualization)

print(f"Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features
print(f"Features: {iris.feature_names}")  # Output: ['sepal length (cm)', 'sepal width (cm)', ...]

# ============================================
# FEATURE SCALING: Important for t-SNE
# ============================================

# t-SNE is sensitive to feature scaling (uses distances)
# Features on different scales distort distance calculations
# Standardize features to have mean=0 and std=1

scaler = StandardScaler()  # StandardScaler normalizes features
X_scaled = scaler.fit_transform(X)
# fit_transform(): Learn scaling from data and apply it
# X_scaled: Features normalized (mean=0, std=1 for each column)

# ============================================
# APPLYING t-SNE
# ============================================

# t-SNE (t-Distributed Stochastic Neighbor Embedding) is a non-linear dimensionality reduction
# Unlike PCA (linear), t-SNE can capture non-linear relationships
# Preserves local neighborhood structure (points close in high-D stay close in low-D)

# TSNE parameters:
# n_components=2: Number of dimensions for embedding (usually 2 or 3 for visualization)
#   - 2D is most common (easy to visualize)
#   - 3D is also possible but harder to visualize
#
# perplexity=30: Effective number of neighbors (typically 5-50)
#   - Lower perplexity = focus on local structure (small neighborhoods)
#   - Higher perplexity = focus on global structure (large neighborhoods)
#   - Default is 30 (good starting point)
#   - Should be less than number of samples
#
# random_state=42: Ensures reproducible results
#   - t-SNE uses random initialization
#   - Same seed = same starting point = same results (mostly)
#
# n_iter=1000: Maximum number of iterations for optimization
#   - More iterations = better optimization but slower
#   - Default is 1000 (usually sufficient)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)

# fit_transform() performs t-SNE:
# 1. Computes pairwise similarities in high-dimensional space (Gaussian)
# 2. Randomly initializes points in low-dimensional space
# 3. Optimizes positions using gradient descent to minimize KL divergence
# 4. Iterates until convergence or max iterations
X_tsne = tsne.fit_transform(X_scaled)
# Returns: Transformed data in 2D space (150 samples × 2 components)

# ============================================
# DISPLAYING t-SNE RESULTS
# ============================================

print(f"\nt-SNE Results:")
print(f"  Reduced shape: {X_tsne.shape}")  # Output: (150, 2) - reduced from 4 to 2 dimensions

# Perplexity used
print(f"  Perplexity: {tsne.perplexity}")  # Perplexity parameter (30)

# KL divergence: Measure of how well low-D distribution matches high-D distribution
print(f"  KL divergence: {tsne.kl_divergence_:.3f}")
# Lower is better (but not directly comparable across different runs)
# This measures how well t-SNE preserved the neighborhood structure

# ============================================
# VISUALIZING t-SNE TRANSFORMATION
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Figure size: 12×5 inches

# Subplot 1: Original data (first 2 features)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot using original features (first 2 features only)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X[:, 0]: First original feature (sepal length)
# X[:, 1]: Second original feature (sepal width)
# c=y: Color by true species labels
# This shows what we'd see if we just picked 2 features arbitrarily

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('Original Data (First 2 Features)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: t-SNE projection (2D embedding)
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot using t-SNE-transformed data
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_tsne[:, 0]: First t-SNE component
# X_tsne[:, 1]: Second t-SNE component
# c=y: Color by true species labels (for comparison)
# This shows the optimized 2D view that preserves local neighborhood structure

plt.xlabel('t-SNE Component 1')  # X-axis: first t-SNE component
plt.ylabel('t-SNE Component 2')  # Y-axis: second t-SNE component
plt.title('t-SNE Projection (2D)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# Interpretation:
# - Left plot: Arbitrary 2D view using first 2 original features
# - Right plot: Optimized 2D view using t-SNE (preserves local neighborhoods)
# - If classes are better separated in right plot, t-SNE found better representation
# - t-SNE focuses on local structure (nearby points stay nearby)
# - Unlike PCA, t-SNE can capture non-linear relationships
# - t-SNE is excellent for visualization but results can vary (non-deterministic)


## Effect of Perplexity

Let's see how perplexity affects t-SNE results.


In [ ]:
# ============================================
# EFFECT OF PERPLEXITY: Understanding This Key Parameter
# ============================================

# Perplexity is the most important t-SNE hyperparameter
# It controls the effective number of neighbors (local vs global structure)
# Lower perplexity = focus on local neighborhoods (small clusters)
# Higher perplexity = focus on global structure (larger clusters)
# We'll test different values to see how they affect results

# Test different perplexity values
perplexities = [5, 15, 30, 50]  # Range from local (5) to global (50)
# - 5: Very local (small neighborhoods, many small clusters)
# - 15: Local (medium neighborhoods)
# - 30: Balanced (default, good starting point)
# - 50: Global (large neighborhoods, fewer clusters)

# Create figure with 2×2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 12))  # 2 rows, 2 columns

# Test each perplexity value
for idx, perp in enumerate(perplexities):
    # Calculate row and column indices for subplot
    row = idx // 2  # Integer division: 0, 0, 1, 1
    col = idx % 2  # Modulo: 0, 1, 0, 1
    
    # Create t-SNE with this perplexity
    tsne_perp = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    # n_components=2: 2D embedding
    # perplexity=perp: Perplexity to test
    # random_state=42: Same seed for fair comparison
    # n_iter=1000: Maximum iterations
    
    # Apply t-SNE
    X_tsne_perp = tsne_perp.fit_transform(X_scaled)  # Transform data
    
    # Plot results
    axes[row, col].scatter(X_tsne_perp[:, 0], X_tsne_perp[:, 1], c=y, 
                          cmap='viridis', s=50, alpha=0.7)
    # X_tsne_perp[:, 0]: First t-SNE component
    # X_tsne_perp[:, 1]: Second t-SNE component
    # c=y: Color by true labels
    
    # Title shows perplexity and KL divergence
    axes[row, col].set_title(f'Perplexity = {perp}\nKL Divergence: {tsne_perp.kl_divergence_:.2f}')
    # KL divergence: Lower is better (but not directly comparable)
    
    axes[row, col].set_xlabel('t-SNE Component 1')  # X-axis label
    axes[row, col].set_ylabel('t-SNE Component 2')  # Y-axis label
    axes[row, col].grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display all plots

# ============================================
# QUANTITATIVE COMPARISON: KL Divergence
# ============================================

print("Effect of Perplexity:")
for perp in perplexities:
    # Create and apply t-SNE with this perplexity
    tsne_perp = TSNE(n_components=2, perplexity=perp, random_state=42, n_iter=1000)
    X_tsne_perp = tsne_perp.fit_transform(X_scaled)  # Transform data
    
    # Display KL divergence
    print(f"  Perplexity {perp}: KL Divergence = {tsne_perp.kl_divergence_:.3f}")
    # KL divergence: Measure of how well low-D matches high-D distribution
    # Lower is better (but values not directly comparable across perplexities)

# Interpretation:
# - Low perplexity (5): Many small clusters, focuses on local structure
# - Medium perplexity (15-30): Balanced, good for most datasets
# - High perplexity (50): Fewer clusters, focuses on global structure
# - Perplexity should be less than number of samples (typically 5-50)
# - Default (30) is usually a good starting point
# - Experiment with different values to see which gives best visualization
# - KL divergence varies with perplexity (not directly comparable)
# - Choose perplexity that gives best cluster separation in visualization


## Comparison with PCA

Let's compare t-SNE with PCA.


In [ ]:
# ============================================
# COMPARING t-SNE WITH PCA: Linear vs Non-Linear
# ============================================

# PCA and t-SNE are both dimensionality reduction techniques but work very differently
# PCA: Linear transformation (finds directions of maximum variance)
# t-SNE: Non-linear transformation (preserves local neighborhoods)
# We'll compare them to see the differences

# Apply PCA for comparison
pca = PCA(n_components=2, random_state=42)
# n_components=2: Reduce to 2 dimensions (same as t-SNE)
# random_state=42: Reproducibility

X_pca = pca.fit_transform(X_scaled)
# Returns: Transformed data in 2D space (150 samples × 2 components)
# PCA finds linear directions of maximum variance

# ============================================
# VISUALIZING COMPARISON: Original vs PCA vs t-SNE
# ============================================

# Create figure with 3 subplots
plt.figure(figsize=(14, 5))  # Figure size: 14×5 inches

# Subplot 1: Original data (first 2 features)
plt.subplot(1, 3, 1)  # 1 row, 3 columns, position 1 (left)

# Scatter plot using original features
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X[:, 0]: First original feature
# X[:, 1]: Second original feature
# c=y: Color by true labels

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('Original Data\n(First 2 Features)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: PCA projection (linear)
plt.subplot(1, 3, 2)  # 1 row, 3 columns, position 2 (middle)

# Scatter plot using PCA-transformed data
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_pca[:, 0]: First principal component
# X_pca[:, 1]: Second principal component
# c=y: Color by true labels

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')  # X-axis: PC1 with variance
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')  # Y-axis: PC2 with variance
plt.title('PCA Projection\n(Linear)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 3: t-SNE projection (non-linear)
plt.subplot(1, 3, 3)  # 1 row, 3 columns, position 3 (right)

# Scatter plot using t-SNE-transformed data
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_tsne[:, 0]: First t-SNE component
# X_tsne[:, 1]: Second t-SNE component
# c=y: Color by true labels

plt.xlabel('t-SNE Component 1')  # X-axis: first t-SNE component
plt.ylabel('t-SNE Component 2')  # Y-axis: second t-SNE component
plt.title('t-SNE Projection\n(Non-linear)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display all plots

# ============================================
# COMPARISON SUMMARY
# ============================================

print("Comparison:")
print(f"  PCA: Linear dimensionality reduction")  # Linear transformation
print(f"  t-SNE: Non-linear dimensionality reduction")  # Non-linear transformation
print(f"  Note: t-SNE better preserves local structure and clusters")
# t-SNE focuses on local neighborhoods (nearby points stay nearby)
# PCA focuses on global variance (directions of maximum spread)

# Interpretation:
# - Original: Arbitrary 2D view (may not show true structure)
# - PCA: Linear projection (finds best linear 2D view)
# - t-SNE: Non-linear projection (finds best non-linear 2D view)
# - For non-linear data, t-SNE often gives better cluster separation
# - For linear data, PCA and t-SNE may give similar results
# - t-SNE is slower but can reveal non-linear patterns
# - PCA is faster and can be applied to new data (has transform method)
# - t-SNE cannot be applied to new data (no transform method - must refit)


## Validation & Testing

Let's validate t-SNE and check for consistency.


In [ ]:
# ============================================
# VALIDATION: Testing Reproducibility
# ============================================

# t-SNE uses random initialization, so results can vary
# With same random_state, results should be identical (reproducible)
# With different random_state, results will differ (but cluster structure should be similar)

# Test reproducibility with same random_state
tsne1 = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne1 = tsne1.fit_transform(X_scaled)  # First run

tsne2 = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_tsne2 = tsne2.fit_transform(X_scaled)  # Second run (same parameters)

# Check if results are identical (they should be with same random_state)
are_identical = np.allclose(X_tsne1, X_tsne2)
# np.allclose(): Check if arrays are approximately equal (within tolerance)
# Should be True if random_state works correctly

print(f"Results with same random_state are identical: {are_identical}")  # Display result

# ============================================
# TESTING WITH DIFFERENT RANDOM STATE
# ============================================

# Test with different random_state (results will differ)
tsne3 = TSNE(n_components=2, perplexity=30, random_state=123, n_iter=1000)
# random_state=123: Different seed (different initialization)
X_tsne3 = tsne3.fit_transform(X_scaled)  # Transform with different initialization

# ============================================
# VISUALIZING REPRODUCIBILITY
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Figure size: 12×5 inches

# Subplot 1: t-SNE with random_state=42
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

plt.scatter(X_tsne1[:, 0], X_tsne1[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_tsne1: Results from first run (random_state=42)
# c=y: Color by true labels

plt.title('t-SNE (random_state=42)')  # Chart title
plt.xlabel('Component 1')  # X-axis label
plt.ylabel('Component 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: t-SNE with random_state=123
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

plt.scatter(X_tsne3[:, 0], X_tsne3[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_tsne3: Results from run with different random_state (123)
# c=y: Color by true labels

plt.title('t-SNE (random_state=123)')  # Chart title
plt.xlabel('Component 1')  # X-axis label
plt.ylabel('Component 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

print("\nNote: t-SNE results vary with different random_state, but cluster structure should be similar")
# Different random_state = different initialization = different final positions
# But cluster structure (which points are together) should be similar
# This is because t-SNE preserves neighborhood structure (not exact positions)

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: Should have 2 components
assert X_tsne.shape[1] == 2, "Should have 2 components"
# X_tsne.shape[1]: Number of columns (should be 2)

# Check 2: KL divergence should be positive
assert tsne.kl_divergence_ > 0, "KL divergence should be positive"
# KL divergence measures how well low-D matches high-D distribution
# Should be positive (perfect match would be 0, but impossible)

print("\n✓ Validation checks passed")  # All checks passed!

# Interpretation:
# - Same random_state = identical results (reproducible)
# - Different random_state = different results (non-deterministic)
# - But cluster structure should be similar (which points are together)
# - This is a key limitation of t-SNE: results are not fully deterministic
# - Always use random_state for reproducibility in research/teaching
# - For production, be aware that results may vary slightly


## Real-World Application

Let's apply t-SNE to a higher-dimensional dataset (Digits).


In [ ]:
# ============================================
# REAL-WORLD APPLICATION: High-Dimensional Dataset
# ============================================

# Digits dataset has 64 features (8×8 pixel images)
# This is high-dimensional data - perfect for demonstrating t-SNE's power
# t-SNE can reveal structure in high-dimensional data that's hard to see otherwise

# Load Digits dataset (handwritten digit images)
digits = load_digits()  # Returns Bunch object
X_digits = digits.data  # Features: pixel values (1797 samples × 64 features)
# Each sample is an 8×8 image flattened to 64 pixels
y_digits = digits.target  # True labels: digit class (0-9, for visualization)

print(f"Digits Dataset Shape: {X_digits.shape}")  # Output: (1797, 64) - 1797 images, 64 pixels
print(f"Number of features: {X_digits.shape[1]}")  # Output: 64 features (pixels)

# ============================================
# FEATURE SCALING: Important for t-SNE
# ============================================

# Standardize features (pixel values)
scaler_digits = StandardScaler()  # Create scaler
X_digits_scaled = scaler_digits.fit_transform(X_digits)
# fit_transform(): Learn scaling and apply it
# X_digits_scaled: Pixel values normalized (mean=0, std=1)

# ============================================
# PCA PREPROCESSING: Speed Optimization
# ============================================

# t-SNE is slow on high-dimensional data (O(n²) complexity)
# Common trick: Use PCA first to reduce dimensions, then apply t-SNE
# This speeds up t-SNE while preserving most information

print("\nApplying PCA first (for speed), then t-SNE...")
# Apply PCA to reduce from 64 to 50 dimensions
pca_digits = PCA(n_components=50, random_state=42)
# n_components=50: Reduce to 50 dimensions (still captures most variance)
# This makes t-SNE much faster (50D instead of 64D)

X_digits_pca = pca_digits.fit_transform(X_digits_scaled)
# Returns: Transformed data (1797 samples × 50 components)
# PCA preserves most variance (50 components from 64 features)

# ============================================
# APPLYING t-SNE TO REDUCED DATA
# ============================================

# Apply t-SNE to PCA-reduced data (much faster!)
tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
# n_components=2: 2D embedding for visualization
# perplexity=30: Default perplexity (good for most datasets)
# random_state=42: Reproducibility

X_digits_tsne = tsne_digits.fit_transform(X_digits_pca)
# Returns: Transformed data in 2D space (1797 samples × 2 components)
# This is the final 2D embedding for visualization

print(f"t-SNE Results:")
print(f"  Reduced shape: {X_digits_tsne.shape}")  # Output: (1797, 2) - reduced from 64 to 2 dimensions
print(f"  KL divergence: {tsne_digits.kl_divergence_:.3f}")  # Quality metric

# ============================================
# VISUALIZING DIGIT SAMPLES
# ============================================

# First, show what the digits look like
plt.figure(figsize=(12, 5))  # Figure size: 12×5 inches

# Display sample digits (first 10)
for i in range(10):
    plt.subplot(2, 5, i+1)  # 2 rows, 5 columns, position i+1
    # digits.images[i]: 8×8 image array (reshaped from 64 pixels)
    plt.imshow(digits.images[i], cmap='gray')  # Display as grayscale image
    plt.title(f'Digit {digits.target[i]}')  # Title shows true digit label
    plt.axis('off')  # Hide axes (cleaner image display)

plt.suptitle('Sample Digits', y=1.02)  # Overall title
plt.tight_layout()
plt.show()  # Display digit samples

# ============================================
# VISUALIZING t-SNE PROJECTION
# ============================================

# Create large figure for t-SNE plot
plt.figure(figsize=(10, 8))  # Figure size: 10×8 inches

# Scatter plot of t-SNE embedding
scatter = plt.scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1], c=y_digits, 
                     cmap='tab10', s=30, alpha=0.6)
# X_digits_tsne[:, 0]: First t-SNE component
# X_digits_tsne[:, 1]: Second t-SNE component
# c=y_digits: Color by digit class (0-9)
# cmap='tab10': Color scheme with 10 distinct colors (one per digit)
# s=30: Smaller point size (many points)
# alpha=0.6: Semi-transparent (easier to see overlapping points)

plt.xlabel('t-SNE Component 1')  # X-axis: first t-SNE component
plt.ylabel('t-SNE Component 2')  # Y-axis: second t-SNE component
plt.title('Digits Dataset: t-SNE Projection (2D)')  # Chart title
plt.colorbar(scatter, label='Digit')  # Color legend showing digit classes (0-9)
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

print("\nNote: t-SNE reveals clear separation between digit classes")
# If t-SNE worked well, different digits should form separate clusters
# This shows that t-SNE can find structure in high-dimensional data (64D → 2D)

# Interpretation:
# - Each point is one handwritten digit image (1797 total)
# - Colors show true digit classes (0-9)
# - If same-colored points cluster together, t-SNE found the right structure
# - Clear separation between digit classes = good embedding
# - Overlapping clusters = digits that look similar (e.g., 3 and 8, 6 and 0)
# - This visualization helps understand relationships between digits
# - t-SNE preserved local structure: similar digits are close together


## Summary & Key Takeaways

### Key Concepts Learned

1. **t-SNE Basics**
   - Non-linear dimensionality reduction
   - Preserves local neighborhood structure
   - Uses t-distribution in low-dimensional space
   - Excellent for visualization

2. **Mathematical Foundation**
   - Models pairwise similarities in high-D (Gaussian)
   - Models pairwise similarities in low-D (t-distribution)
   - Minimizes KL divergence between distributions
   - Gradient descent optimization

3. **Key Hyperparameters**
   - **perplexity**: Number of effective neighbors (5-50)
   - **n_components**: Output dimensions (usually 2 or 3)
   - **learning_rate**: Optimization step size
   - **n_iter**: Maximum iterations

4. **Best Practices**
   - Use PCA first for high-dimensional data (speed)
   - Standardize features before t-SNE
   - Experiment with perplexity (start with 30)
   - Use random_state for reproducibility
   - Results are non-deterministic (vary with initialization)

### When to Use t-SNE

✅ **Good for:**
- Visualizing high-dimensional data
- Exploring cluster structure
- Non-linear data relationships
- Understanding data topology
- Embedding visualization
- Small to medium datasets (< 10,000 samples)

❌ **Not ideal for:**
- Very large datasets (computationally expensive O(n²))
- When you need to transform new data (no transform method)
- When global structure is important
- When deterministic results are required
- Real-time applications (slow)
- When linear methods (PCA) suffice

### Next Steps

- Compare with **UMAP** (faster alternative)
- Use **PCA + t-SNE** for high-dimensional data
- Try **3D t-SNE** for better visualization
- Explore **Barnes-Hut t-SNE** for faster computation
- Apply to **word embeddings** and **image features**
